# 05 Evaluation and Ablation

This notebook evaluates all saved models on the same fixed test candidate set. It computes HR@10, NDCG@10, Recall@10, improvement over baseline, and automatic interpretation text.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch

from src import config
from src.preprocessing import build_metadata_from_mappings
from src.models import build_model
from src.evaluate import evaluate_ranking
from src.analysis import append_or_replace_metrics, plot_model_comparison, plot_ablation_results
from src.utils import load_json, ensure_dir, set_seed

set_seed()
ensure_dir(config.RESULTS_DIR)
ensure_dir(config.FIGURES_DIR)

In [ ]:
test_candidates = pd.read_csv(config.PROCESSED_DATA_DIR / 'test_candidates.csv')
mappings = load_json(config.PROCESSED_DATA_DIR / 'mappings.json')
metadata = build_metadata_from_mappings(mappings)

feature_names = {
    'baseline': 'user_id + movie_id',
    'time': 'user_id + movie_id + time context',
    'user': 'user_id + movie_id + user context',
    'movie': 'user_id + movie_id + movie context',
    'full': 'user_id + movie_id + time + user + movie context',
}

## Test Evaluation

Every model is scored on the same candidate rows. This makes the comparison fair.

In [ ]:
rows = []
full_per_group = None
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for variant in ['baseline', 'time', 'user', 'movie', 'full']:
    model_path = config.MODEL_SAVE_PATHS[variant]
    model = build_model(variant, metadata)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    metrics, per_group, scored = evaluate_ranking(model, test_candidates, metadata, variant, k=config.TOP_K)
    rows.append({'split': 'test', 'model': variant, 'used_features': feature_names[variant], **metrics})
    if variant == 'full':
        full_per_group = per_group
        per_group.to_csv(config.RESULTS_DIR / 'full_model_test_per_group.csv', index=False)
        scored.to_csv(config.RESULTS_DIR / 'full_model_test_scored_candidates.csv', index=False)

metrics_df = append_or_replace_metrics(config.RESULTS_DIR / 'metrics.csv', rows)
test_metrics = pd.DataFrame(rows)
display(test_metrics)

## Ablation Table

Deltas are computed against the baseline model on the same test candidate set.

In [ ]:
baseline = test_metrics[test_metrics['model'] == 'baseline'].iloc[0]
ablation = test_metrics.copy()
ablation['delta_HR@10_vs_baseline'] = ablation['HR@10'] - baseline['HR@10']
ablation['delta_NDCG@10_vs_baseline'] = ablation['NDCG@10'] - baseline['NDCG@10']
ablation['delta_Recall@10_vs_baseline'] = ablation['Recall@10'] - baseline['Recall@10']

best_context_model = ablation[ablation['model'].isin(['time', 'user', 'movie'])].sort_values('delta_NDCG@10_vs_baseline', ascending=False).iloc[0]['model']

def interpret(row):
    if row['model'] == 'baseline':
        return 'Reference model without explicit context.'
    if row['model'] == 'full':
        return 'The full context-aware model improves ranking quality compared to baseline.' if row['delta_NDCG@10_vs_baseline'] > 0 else 'The full context-aware model does not improve over baseline on this test set.'
    if row['model'] == best_context_model and row['delta_NDCG@10_vs_baseline'] > 0:
        label = {'time': 'Temporal context', 'user': 'User demographics', 'movie': 'Movie metadata'}[row['model']]
        return f'{label} appears to be the strongest single context signal.'
    if row['delta_NDCG@10_vs_baseline'] > 0:
        return 'This context group provides a measurable improvement over baseline.'
    return 'This context group does not improve NDCG@10 over baseline; the signal may be weak or noisy.'

ablation['interpretation'] = ablation.apply(interpret, axis=1)
ablation.to_csv(config.RESULTS_DIR / 'ablation_results.csv', index=False)
display(ablation[['model', 'used_features', 'HR@10', 'NDCG@10', 'Recall@10', 'delta_HR@10_vs_baseline', 'delta_NDCG@10_vs_baseline', 'interpretation']])

In [ ]:
plot_model_comparison(test_metrics, 'HR@10', config.FIGURES_DIR / 'model_comparison_hr10.png')
plot_model_comparison(test_metrics, 'NDCG@10', config.FIGURES_DIR / 'model_comparison_ndcg10.png')
plot_ablation_results(ablation, config.FIGURES_DIR / 'ablation_ndcg10.png')
print('Saved comparison plots to', config.FIGURES_DIR)

The ablation table directly answers which context type helps most and whether the full context-aware model improves over the baseline.